<a href="https://colab.research.google.com/github/nikhitarao/nikhitarao_projects/blob/Healthcare-Insurance-Medical-Inflation-Analysis-using-Synthetic-Data/Inpatient_Healthcare_Medical_Procedure_Synthetic_Data_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Code to generate initial 100,000 records for medical claims procedures data for New York State

#### Import Libraries

In [13]:
import os
import random
import dask
import dask.dataframe as dd
import numpy as np
import pandas as pd
from dask import delayed

#### Define parameters for synthetic data generation

In [14]:
# Parameters for synthetic data generation
num_records = 100000
inpatient_procedures = [
    'Knee Replacement Surgery', 'Hip Replacement Surgery', 'Heart Bypass Surgery',
    'Spinal Fusion Surgery', 'Gallbladder Removal', 'Appendectomy', 'Hysterectomy',
    'Cataract Surgery', 'Tonsillectomy', 'Liver Transplant'
]
medicine_types = ['Prescription', 'Over-the-Counter', 'Specialized', 'Generic', 'Vaccine']
hospital_provider_types = ['General Hospital', 'Specialty Clinic', 'Urgent Care', 'Primary Care', 'Pharmacy']
location_types = ['Urban', 'Suburban', 'Rural']
age_groups = ['Infant', 'Child', 'Adult', 'Senior']
genders = ['Male', 'Female', 'Other']
insurance_types = ['Private', 'Government', 'Uninsured', 'Medicare', 'Medicaid']
income_brackets = ['Low', 'Middle', 'High']
education_levels = ['High School', 'Bachelor\'s', 'Master\'s', 'Doctorate', 'Other']
seasons = ['Summer', 'Winter', 'Fall', 'Spring']
days_of_week = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
chronic_conditions = [
    'None', 'Asthma', 'Diabetes', 'Hypertension', 'Chronic Kidney Disease',
    'Arthritis', 'Chronic Obstructive Pulmonary Disease', 'Depression', 'Heart Disease'
]

#### Helper function for adjusting costs and length of stay

In [15]:
# Helper function to adjust costs and lengths of stay
def adjust_for_conditions(condition, base_cost, length_of_stay):
    if condition == 'None':
        return base_cost, length_of_stay
    cost_multiplier = {
        'Asthma': 1.1, 'Diabetes': 1.2, 'Hypertension': 1.15,
        'Chronic Kidney Disease': 1.5, 'Arthritis': 1.1,
        'Chronic Obstructive Pulmonary Disease': 1.3, 'Depression': 1.2,
        'Heart Disease': 1.5
    }
    stay_multiplier = {
        'Asthma': 1.1, 'Diabetes': 1.2, 'Hypertension': 1.15,
        'Chronic Kidney Disease': 1.5, 'Arthritis': 1.1,
        'Chronic Obstructive Pulmonary Disease': 1.3, 'Depression': 1.2,
        'Heart Disease': 1.5
    }
    adjusted_cost = base_cost * cost_multiplier.get(condition, 1)
    adjusted_stay = length_of_stay * stay_multiplier.get(condition, 1)
    return adjusted_cost, adjusted_stay

#### Define the synthetic data function

In [16]:
# Define your synthetic data generation function
def generate_synthetic_data_chunk(num_records):
    years = np.random.choice(range(2010, 2024), num_records)  # Year range
    chronic_condition_list = np.random.choice(
        chronic_conditions, num_records,
        p=[0.5, 0.1, 0.1, 0.1, 0.05, 0.05, 0.03, 0.03, 0.04]
    )
    base_cost = np.random.uniform(5000, 100000, num_records)
    length_of_stay = np.random.normal(5, 2, num_records).clip(1, 30)

    adjusted_costs_and_stays = [
        adjust_for_conditions(cond, cost, stay)
        for cond, cost, stay in zip(chronic_condition_list, base_cost, length_of_stay)
    ]
    adjusted_costs, adjusted_stays = zip(*adjusted_costs_and_stays)

    data = {
        'Year': years,
        'Season': np.random.choice(seasons, num_records),
        'Day of the Week': np.random.choice(days_of_week, num_records, p=[0.15, 0.15, 0.15, 0.15, 0.15, 0.15, 0.1]),
        'State': ['New York'] * num_records,
        'Urbanization Level (%)': np.random.uniform(50, 100, num_records).round(2),
        'Inpatient Procedure Type': np.random.choice(inpatient_procedures, num_records),
        'Medicine Type': np.random.choice(medicine_types, num_records),
        'Hospital Provider Type': np.random.choice(hospital_provider_types, num_records),
        'Location Type': np.random.choice(location_types, num_records),
        'Patient Age Group': np.random.choice(age_groups, num_records, p=[0.05, 0.1, 0.5, 0.35]),
        'Gender': np.random.choice(genders, num_records, p=[0.49, 0.49, 0.02]),
        'Insurance Type': np.random.choice(insurance_types, num_records, p=[0.4, 0.3, 0.05, 0.15, 0.1]),
        'Household Income Bracket': np.random.choice(income_brackets, num_records, p=[0.3, 0.5, 0.2]),
        'Education Level': np.random.choice(education_levels, num_records, p=[0.4, 0.3, 0.2, 0.05, 0.05]),
        'Chronic Conditions': chronic_condition_list,
        'Base Cost ($)': base_cost.round(2),
        'Length of Stay (days)': np.array(adjusted_stays).round(0),
    }

     # Set deductible to 0 for uninsured
    data['Deductible Amount ($)'] = [
        0 if ins == 'Uninsured' else np.clip(np.random.normal(1500, 500), 500, 5000).round(2)
        for ins in data['Insurance Type']
    ]

    # Conditional adjustments for Medicare/Medicaid
    data['Medicare Eligibility'] = [
        1 if ins == 'Medicare' and age == 'Senior' else 0
        for ins, age in zip(data['Insurance Type'], data['Patient Age Group'])
    ]
    data['Medicaid Eligibility'] = [
        1 if ins == 'Medicaid' and income == 'Low' else 0
        for ins, income in zip(data['Insurance Type'], data['Household Income Bracket'])
    ]

    # Adjusted out-of-pocket cost calculation
    data['Co-Payment (%)'] = np.random.normal(20, 5, num_records).clip(5, 50).round(2)
    data['Adjusted Out-of-Pocket Cost ($)'] = [
        max(0, base_cost - deductible) * (copay / 100)
        for base_cost, deductible, copay in zip(
            data['Base Cost ($)'], data['Deductible Amount ($)'], data['Co-Payment (%)']
        )
    ]

    # Calculated feature: Adjusted Cost
    data['Adjusted Cost ($)'] = (
        np.array(adjusted_costs) * (1 + np.random.uniform(2, 8, num_records) / 100)
    ).round(2)

    return pd.DataFrame(data)

#### Dask function to generate records in parallel

In [17]:
# Function to generate synthetic data in parallel
def generate_synthetic_data_dask(num_records, chunks):
    records_per_chunk = num_records // chunks

    # Generate data for each chunk using Dask Delayed
    delayed_chunks = [
        delayed(generate_synthetic_data_chunk)(records_per_chunk)
        for _ in range(chunks)
    ]

    # Convert to Dask DataFrame
    dask_df = dd.from_delayed(delayed_chunks)
    return dask_df

#### Calling code to generate synthetic data

In [18]:
total_records = 1000000
chunks = 10  # Number of parallel chunks

# Generate synthetic data
synthetic_data = generate_synthetic_data_dask(total_records, chunks)

# Compute the result as a Pandas DataFrame
df_with_conditions = synthetic_data.compute()

#### Save to File Path

In [20]:
# Save to CSV
file_path = 'C:/Users/nikhi/OneDrive/Documents/GitHub/nikhitarao_projects/initial_synthetic_inpatient_healthcare_medical_procedure_dataset.csv'

# Create the directory if it doesn't exist
os.makedirs(os.path.dirname(file_path), exist_ok=True)

df_with_conditions.to_csv(file_path, index=False)

df_with_conditions.head()

,Year,Season,Day of the Week,State,Urbanization Level (%),Inpatient Procedure Type,Medicine Type,Hospital Provider Type,Location Type,Patient Age Group,...,Education Level,Chronic Conditions,Base Cost ($),Length of Stay (days),Deductible Amount ($),Medicare Eligibility,Medicaid Eligibility,Co-Payment (%),Adjusted Out-of-Pocket Cost ($),Adjusted Cost ($)
0,2018,Winter,Friday,New York,73.46,Appendectomy,Generic,Primary Care,Urban,Infant,...,Master's,None,17855.58,4.0,1499.47,0,0,16.99,2778.903089,19053.13
1,2015,Spring,Wednesday,New York,55.84,Liver Transplant,Generic,General Hospital,Rural,Adult,...,Master's,None,15440.59,5.0,0.00,0,0,27.83,4297.116197,16299.16
2,2012,Fall,Wednesday,New York,68.37,Hysterectomy,Over-the-Counter,Specialty Clinic,Rural,Child,...,High School,Asthma,15054.73,6.0,1703.13,0,0,18.22,2432.661520,16902.88
3,2014,Fall,Monday,New York,81.90,Knee Replacement Surgery,Specialized,Specialty Clinic,Rural,Adult,...,High School,None,69220.47,7.0,1632.86,0,0,20.75,14024.429075,71301.41
4,2015,Summer,Sunday,New York,59.40,Appendectomy,Prescription,Pharmacy,Rural,Adult,...,Master's,Asthma,64586.90,8.0,1745.80,0,0,22.37,14057.554070,75179.62
